# Case-Based Triage with Vector Search

Emergency department triage prediction using Qdrant vector similarity search.
Instead of a traditional classifier, we embed patient presentations into a vector space
and predict acuity from the K most similar historical cases.

**Approach:**
1. Serialize each patient record (chief complaint, vitals, history) into natural language
2. Embed using nomic-embed-text-v1.5 (strong on clinical text, zero-config via Fastembed)
3. Store in Qdrant with structured payloads for filtered retrieval
4. Predict acuity via distance-weighted KNN over the vector space
5. Analyze prediction fairness across demographic groups

In [ ]:
# Install dependencies (Kaggle notebook compatible)
!pip install -q "qdrant-client[fastembed]>=1.12.0" python-dotenv tqdm seaborn

In [ ]:
import os
import sys
import time
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score
)
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

sns.set_theme(style="whitegrid")
print("imports ok")

## 1. Configuration

In [ ]:
# Paths: adjust if running on Kaggle vs local
KAGGLE = os.path.exists("/kaggle/input")
if KAGGLE:
    DATA_DIR = Path("/kaggle/input/triagegeist")
else:
    DATA_DIR = Path("../data")

# Model
EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v1.5"
EMBEDDING_DIM = 768

# Search
K_NEIGHBORS = 15
COLLECTION_NAME = "triagegeist"

# ESI labels
ESI_LABELS = {1: "Resuscitation", 2: "Emergent", 3: "Urgent", 4: "Less Urgent", 5: "Non-Urgent"}

# Demographic columns for bias analysis
DEMOGRAPHIC_COLS = ["age_group", "sex", "language", "insurance_type"]

print(f"data dir: {DATA_DIR}")
print(f"files: {list(DATA_DIR.glob('*.csv')) if DATA_DIR.exists() else 'NOT FOUND'}")

## 2. Load and Merge Data

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
complaints = pd.read_csv(DATA_DIR / "chief_complaints.csv")
history = pd.read_csv(DATA_DIR / "patient_history.csv")

# Merge
train = train.merge(complaints, on="patient_id", how="left").merge(history, on="patient_id", how="left")
test = test.merge(complaints, on="patient_id", how="left").merge(history, on="patient_id", how="left")

print(f"train: {train.shape}, test: {test.shape}")
print(f"\nacuity distribution:")
print(train["triage_acuity"].value_counts().sort_index())

In [ ]:
# Quick EDA: acuity distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train["triage_acuity"].value_counts().sort_index().plot.bar(ax=axes[0], color="#3498db")
axes[0].set_title("Triage Acuity Distribution")
axes[0].set_xlabel("ESI Level")
axes[0].set_ylabel("Count")

# Missing vitals by acuity
vitals = ["systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate", "temperature_c", "spo2"]
missing_by_acuity = train.groupby("triage_acuity")[vitals].apply(lambda x: x.isna().mean()).T
sns.heatmap(missing_by_acuity, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Missing Vitals Rate by Acuity")
plt.tight_layout()
plt.show()

## 3. Serialize Patients to Text

In [ ]:
HISTORY_PREFIX = "hx_"

def serialize_patient(row):
    """Convert a patient row into natural language for embedding."""
    parts = []

    # Chief complaint
    cc = row.get("chief_complaint_raw")
    if pd.notna(cc) and str(cc).strip():
        parts.append(f"Chief complaint: {cc.strip()}")

    complaint_system = row.get("chief_complaint_system")
    if pd.notna(complaint_system):
        parts.append(f"System: {complaint_system}")

    mental = row.get("mental_status_triage")
    if pd.notna(mental):
        parts.append(f"Mental status: {mental}")

    # Vitals
    vitals_map = {
        "systolic_bp": "systolic BP {:.0f}",
        "diastolic_bp": "diastolic BP {:.0f}",
        "heart_rate": "heart rate {:.0f}",
        "respiratory_rate": "respiratory rate {:.0f}",
        "temperature_c": "temperature {:.1f}C",
        "spo2": "SpO2 {:.0f}%",
        "gcs_total": "GCS {:.0f}",
        "pain_score": "pain {:.0f}/10",
        "shock_index": "shock index {:.2f}",
        "news2_score": "NEWS2 {:.0f}",
        "bmi": "BMI {:.1f}",
    }
    vitals_parts = []
    for col, fmt in vitals_map.items():
        val = row.get(col)
        if pd.notna(val) and val != -1:
            vitals_parts.append(fmt.format(val))
    if vitals_parts:
        parts.append(f"Vitals: {', '.join(vitals_parts)}")

    # Context
    for col, fmt in [("arrival_mode", "Arrival: {}"), ("pain_location", "Pain location: {}"), ("transport_origin", "Transport from: {}")]:
        val = row.get(col)
        if pd.notna(val):
            parts.append(fmt.format(val))

    # Demographics
    age, sex = row.get("age"), row.get("sex")
    if pd.notna(age) and pd.notna(sex):
        parts.append(f"{int(age)} year old {sex}")

    # History
    hx_cols = [c for c in row.index if c.startswith(HISTORY_PREFIX)]
    active_hx = [c.replace(HISTORY_PREFIX, "").replace("_", " ") for c in hx_cols if row.get(c) == 1]
    if active_hx:
        parts.append(f"History: {', '.join(active_hx)}")

    # Utilization
    prior_ed = row.get("num_prior_ed_visits_12m")
    if pd.notna(prior_ed) and prior_ed > 0:
        parts.append(f"{int(prior_ed)} ED visits in past year")
    num_meds = row.get("num_active_medications")
    if pd.notna(num_meds) and num_meds > 0:
        parts.append(f"{int(num_meds)} active medications")

    return ". ".join(parts)


# Test serialization
sample = train.iloc[0]
print(f"ESI-{sample['triage_acuity']}:")
print(serialize_patient(sample))

In [ ]:
# Serialize all patients
print("serializing train...")
train["text"] = [serialize_patient(row) for _, row in tqdm(train.iterrows(), total=len(train))]

print("serializing test...")
test["text"] = [serialize_patient(row) for _, row in tqdm(test.iterrows(), total=len(test))]

print(f"\nsample texts by acuity:")
for esi in [1, 2, 3, 4, 5]:
    row = train[train["triage_acuity"] == esi].iloc[0]
    print(f"\nESI-{esi}: {row['text'][:200]}...")

## 4. Split, Embed, and Index in Qdrant

We use a train/validation split for evaluation. The "index" set goes into Qdrant,
the "query" set is used to evaluate retrieval-based predictions.

In [ ]:
# Stratified split: 80% index (stored in Qdrant), 20% query (for evaluation)
train_idx, val_idx = train_test_split(
    train.index, test_size=0.2, stratify=train["triage_acuity"], random_state=42
)
index_df = train.loc[train_idx].reset_index(drop=True)
query_df = train.loc[val_idx].reset_index(drop=True)

print(f"index set: {len(index_df)} patients")
print(f"query set: {len(query_df)} patients")
print(f"\nindex acuity distribution:")
print(index_df["triage_acuity"].value_counts().sort_index())

In [ ]:
# Initialize embedding model
print(f"loading {EMBEDDING_MODEL}...")
embed_model = TextEmbedding(model_name=EMBEDDING_MODEL)
print("model loaded")

In [ ]:
# Embed the index set
print(f"embedding {len(index_df)} index records...")
t0 = time.perf_counter()
index_embeddings = list(embed_model.embed(index_df["text"].tolist(), batch_size=128))
elapsed = time.perf_counter() - t0
print(f"embedded in {elapsed:.1f}s ({len(index_df)/elapsed:.0f} records/s)")
print(f"embedding shape: {index_embeddings[0].shape}")

In [ ]:
# Start Qdrant in-memory (no Docker needed on Kaggle)
client = QdrantClient(":memory:")

# Create collection
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIM,
        distance=models.Distance.COSINE,
    ),
)

# Payload indices for filtered search
for field in ["triage_acuity", "age_group", "sex", "insurance_type", "chief_complaint_system"]:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=models.PayloadSchemaType.KEYWORD,
    )

print(f"collection '{COLLECTION_NAME}' created")

In [ ]:
# Build payloads
payload_cols = [
    "patient_id", "triage_acuity", "chief_complaint_raw", "chief_complaint_system",
    "mental_status_triage", "disposition", "ed_los_hours",
    "systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate",
    "temperature_c", "spo2", "gcs_total", "pain_score", "shock_index", "news2_score",
    "age", "age_group", "sex", "language", "insurance_type",
    "arrival_mode", "num_comorbidities",
]
payload_cols = [c for c in payload_cols if c in index_df.columns]

payloads = []
for _, row in index_df.iterrows():
    p = {}
    for col in payload_cols:
        val = row[col]
        if pd.notna(val):
            p[col] = int(val) if isinstance(val, float) and val == int(val) else val
        else:
            p[col] = None
    payloads.append(p)

# Upload
print(f"uploading {len(index_embeddings)} vectors...")
t0 = time.perf_counter()
client.upload_collection(
    collection_name=COLLECTION_NAME,
    vectors=[e.tolist() for e in index_embeddings],
    payload=payloads,
    ids=index_df["patient_id"].tolist(),
)
elapsed = time.perf_counter() - t0
print(f"uploaded in {elapsed:.1f}s")

info = client.get_collection(COLLECTION_NAME)
print(f"collection points: {info.points_count}")

## 5. Predict Triage Acuity via KNN Search

In [ ]:
def predict_knn(query_texts, query_patient_ids, k=K_NEIGHBORS):
    """Predict acuity via distance-weighted KNN over Qdrant."""
    embeddings = list(embed_model.embed(query_texts, batch_size=128))
    predictions = []

    for emb, pid in tqdm(zip(embeddings, query_patient_ids), total=len(embeddings), desc="predicting"):
        hits = client.query_points(
            collection_name=COLLECTION_NAME,
            query=emb.tolist(),
            limit=k,
            with_payload=True,
            query_filter=models.Filter(
                must_not=[
                    models.FieldCondition(
                        key="patient_id",
                        match=models.MatchValue(value=pid),
                    )
                ]
            ),
        )

        acuities, scores = [], []
        for hit in hits.points:
            a = hit.payload.get("triage_acuity")
            if a is not None:
                acuities.append(int(a))
                scores.append(hit.score)

        if not acuities:
            predictions.append({"patient_id": pid, "predicted_acuity": 3, "confidence": 0.0})
            continue

        # Distance-weighted voting
        weights = np.array(scores)
        votes = {}
        for a, w in zip(acuities, weights):
            votes[a] = votes.get(a, 0) + w
        predicted = max(votes, key=votes.get)
        total = sum(votes.values())
        confidence = votes[predicted] / total if total > 0 else 0.0

        predictions.append({
            "patient_id": pid,
            "predicted_acuity": predicted,
            "confidence": round(confidence, 3),
            "neighbor_acuities": acuities,
        })

    return pd.DataFrame(predictions)


# Predict on validation set
print(f"predicting {len(query_df)} validation patients...")
t0 = time.perf_counter()
pred_df = predict_knn(query_df["text"].tolist(), query_df["patient_id"].tolist())
elapsed = time.perf_counter() - t0
print(f"predicted in {elapsed:.1f}s ({len(query_df)/elapsed:.0f} patients/s)")

## 6. Evaluation

In [ ]:
# Merge predictions with ground truth
eval_df = query_df[["patient_id", "triage_acuity"] + DEMOGRAPHIC_COLS].merge(pred_df, on="patient_id")

y_true = eval_df["triage_acuity"]
y_pred = eval_df["predicted_acuity"]

# Core metrics
print(f"accuracy:       {accuracy_score(y_true, y_pred):.3f}")
print(f"f1 (macro):     {f1_score(y_true, y_pred, average='macro'):.3f}")
print(f"f1 (weighted):  {f1_score(y_true, y_pred, average='weighted'):.3f}")
print(f"cohen kappa:    {cohen_kappa_score(y_true, y_pred, weights='quadratic'):.3f}")
print(f"within 1 level: {np.mean(np.abs(y_true - y_pred) <= 1):.3f}")
print()
print(classification_report(
    y_true, y_pred,
    target_names=[f"ESI-{k} ({v})" for k, v in ESI_LABELS.items()],
))

In [ ]:
# Confusion matrix
labels = sorted(y_true.unique())
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=[f"ESI-{l}" for l in labels],
            yticklabels=[f"ESI-{l}" for l in labels])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix (counts)")

sns.heatmap(cm_pct, annot=True, fmt=".2f", cmap="Blues", ax=axes[1],
            xticklabels=[f"ESI-{l}" for l in labels],
            yticklabels=[f"ESI-{l}" for l in labels])
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].set_title("Confusion Matrix (row-normalized)")

plt.tight_layout()
plt.show()

## 7. Bias Analysis

Does the system perform equitably across demographic groups?
We measure accuracy, over-triage (predicted more severe than actual),
and under-triage (predicted less severe) by age, sex, language, and insurance.

In [ ]:
def bias_analysis(df, y_true_col="triage_acuity", y_pred_col="predicted_acuity"):
    results = {}
    for demo_col in DEMOGRAPHIC_COLS:
        if demo_col not in df.columns:
            continue
        group_metrics = []
        for group_val, group_df in df.groupby(demo_col):
            if len(group_df) < 10:
                continue
            yt = group_df[y_true_col]
            yp = group_df[y_pred_col]
            diff = yp - yt
            group_metrics.append({
                "group": str(group_val),
                "n": len(group_df),
                "accuracy": accuracy_score(yt, yp),
                "f1_weighted": f1_score(yt, yp, average="weighted", zero_division=0),
                "over_triage_rate": float(np.mean(diff < 0)),
                "under_triage_rate": float(np.mean(diff > 0)),
                "mean_error": float(diff.mean()),
            })
        results[demo_col] = group_metrics

        # Print table
        print(f"\n{'='*65}")
        print(f"BIAS ANALYSIS: {demo_col}")
        print(f"{'='*65}")
        print(f"{'group':<20} {'n':>6} {'acc':>6} {'f1':>6} {'over':>6} {'under':>6}")
        print("-" * 65)
        for g in sorted(group_metrics, key=lambda x: -x['n']):
            print(f"{g['group']:<20} {g['n']:>6} {g['accuracy']:>6.3f} {g['f1_weighted']:>6.3f} {g['over_triage_rate']:>6.3f} {g['under_triage_rate']:>6.3f}")

    return results

bias_results = bias_analysis(eval_df)

In [ ]:
# Visualize bias
n_dims = len(bias_results)
fig, axes = plt.subplots(1, n_dims, figsize=(5 * n_dims, 5))
if n_dims == 1:
    axes = [axes]

for ax, (demo_col, groups) in zip(axes, bias_results.items()):
    groups_sorted = sorted(groups, key=lambda x: -x["accuracy"])
    names = [g["group"] for g in groups_sorted]
    accs = [g["accuracy"] for g in groups_sorted]
    colors = ["#e74c3c" if g["under_triage_rate"] > 0.3 else "#3498db" for g in groups_sorted]

    ax.barh(names, accs, color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Accuracy")
    ax.set_title(f"Accuracy by {demo_col}")
    ax.axvline(x=np.mean(accs), color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 8. Explainability: Case-Based Reasoning

For any patient, we can show the K most similar historical cases that
informed the prediction. This is the core clinical value proposition.

In [ ]:
def explain_prediction(row, k=5):
    text = serialize_patient(row)
    embedding = list(embed_model.embed([text]))[0]
    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=embedding.tolist(),
        limit=k,
        with_payload=True,
    )
    print(f"Patient: {text[:150]}...")
    print(f"Actual acuity: ESI-{row.get('triage_acuity', '?')}")
    print(f"\nTop {k} similar cases:")
    print("-" * 80)
    for i, hit in enumerate(hits.points):
        p = hit.payload
        print(f"  {i+1}. [score={hit.score:.3f}] ESI-{p.get('triage_acuity')} | "
              f"{p.get('chief_complaint_raw', 'N/A')[:60]} | "
              f"{p.get('age_group', '?')}, {p.get('sex', '?')} | "
              f"disposition: {p.get('disposition', '?')}")

# Show a few examples
for esi in [1, 3, 5]:
    sample = query_df[query_df["triage_acuity"] == esi].iloc[0]
    explain_prediction(sample)
    print("\n")

## 9. Generate Test Predictions

For the competition submission: re-index ALL training data, then predict on the test set.

In [ ]:
# Re-create collection with full training data
FULL_COLLECTION = "triagegeist_full"

client.create_collection(
    collection_name=FULL_COLLECTION,
    vectors_config=models.VectorParams(size=EMBEDDING_DIM, distance=models.Distance.COSINE),
)

# Embed all training data
print(f"embedding all {len(train)} training records...")
t0 = time.perf_counter()
all_train_embeddings = list(embed_model.embed(train["text"].tolist(), batch_size=128))
print(f"embedded in {time.perf_counter() - t0:.1f}s")

# Build payloads and upload
all_payloads = []
for _, row in train.iterrows():
    p = {}
    for col in payload_cols:
        if col in train.columns:
            val = row[col]
            p[col] = (int(val) if isinstance(val, float) and val == int(val) else val) if pd.notna(val) else None
    all_payloads.append(p)

client.upload_collection(
    collection_name=FULL_COLLECTION,
    vectors=[e.tolist() for e in all_train_embeddings],
    payload=all_payloads,
    ids=train["patient_id"].tolist(),
)
print(f"uploaded {len(train)} points to '{FULL_COLLECTION}'")

In [ ]:
# Predict on test set
print(f"embedding {len(test)} test records...")
test_embeddings = list(embed_model.embed(test["text"].tolist(), batch_size=128))

test_predictions = []
for emb, pid in tqdm(zip(test_embeddings, test["patient_id"].tolist()), total=len(test), desc="test predict"):
    hits = client.query_points(
        collection_name=FULL_COLLECTION,
        query=emb.tolist(),
        limit=K_NEIGHBORS,
        with_payload=True,
    )
    acuities, scores = [], []
    for hit in hits.points:
        a = hit.payload.get("triage_acuity")
        if a is not None:
            acuities.append(int(a))
            scores.append(hit.score)

    if not acuities:
        test_predictions.append({"patient_id": pid, "triage_acuity": 3})
        continue

    weights = np.array(scores)
    votes = {}
    for a, w in zip(acuities, weights):
        votes[a] = votes.get(a, 0) + w
    test_predictions.append({"patient_id": pid, "triage_acuity": max(votes, key=votes.get)})

submission = pd.DataFrame(test_predictions)
submission.to_csv("submission.csv", index=False)
print(f"\nsubmission shape: {submission.shape}")
print(submission["triage_acuity"].value_counts().sort_index())
print("\nsaved submission.csv")

## 10. Summary

This notebook demonstrates **case-based triage prediction** using vector similarity search.

**Key design decisions:**
- Patient records are serialized as natural language (combining chief complaint, vitals, demographics, and history) so that a text embedding model can capture the full clinical picture in a single vector.
- Qdrant's in-memory mode runs the entire pipeline without external infrastructure.
- Payload filtering enables demographic-sliced retrieval for bias analysis.
- Distance-weighted KNN voting accounts for varying similarity to neighbors.
- Every prediction is explainable: "these K past cases drove this acuity assessment."

**Clinical relevance:** This mirrors how experienced ED physicians triage: pattern matching against recalled similar presentations. The system makes that process transparent and auditable.

**Limitations:**
- Performance depends heavily on embedding model quality for clinical text.
- The serialization format is hand-crafted; optimal text representation is an open question.
- KNN search time scales with query volume (though Qdrant's HNSW index keeps it sub-linear).
- Synthetic data may not capture the full complexity of real ED presentations.